# 💎 Diamond Dynamics — EDA, Modelling, and Market Segmentation

## Project Overview
This notebook documents the complete machine learning workflow for **Diamond Dynamics**, an AI-powered project built for **diamond price prediction** and **market segmentation**.

The project solves two practical retail analytics problems:
- **Regression problem:** predict diamond price in INR using physical and quality attributes.
- **Clustering problem:** group diamonds into meaningful market segments for inventory strategy and buyer targeting.

## Business Context
Diamond pricing is influenced by multiple factors such as:
- Carat
- Cut
- Color
- Clarity
- Physical dimensions

Because these factors interact in a non-linear way, machine learning is useful for:
- Price estimation
- Inventory positioning
- Segment discovery
- Recommendation systems

## What this notebook covers
1. Data loading and inspection
2. Data preprocessing and cleaning
3. Outlier detection and treatment
4. Skewness detection and log transformation
5. Exploratory Data Analysis (EDA)
6. Feature engineering
7. Ordinal encoding
8. Feature selection
9. Regression modelling with 5 ML models + ANN
10. Clustering with KMeans + PCA
11. Saving models and metadata for deployment

## Notes
This notebook is structured to demonstrate:
- Clear problem understanding
- End-to-end pipeline thinking
- Feature engineering depth
- Practical model comparison
- Explainable clustering
- Deployment readiness through saved artifacts

In [ ]:
!pip install -q gdown joblib xgboost tensorflow statsmodels seaborn plotly

## 1. Import libraries and define project configuration

In this section, I import all required libraries and define the global configuration used throughout the notebook.

This includes:
- Data manipulation libraries
- Visualization libraries
- Machine learning libraries
- Deep learning tools for ANN
- Clustering and PCA tools
- Artifact saving utilities

I also define project constants such as:
- Dataset URL
- Folder paths
- Random seed
- USD to INR conversion rate

In [ ]:
import os
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import gdown
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_STATE = 42
USD_TO_INR = 83.5
GOOGLE_DRIVE_URL = "https://drive.google.com/uc?id=1m9HU-CoGXCzLtj9DyAoZt-13BfHKQH0c"

DATA_DIR = Path("data")
PLOTS_DIR = Path("plots")
MODELS_DIR = Path("models")
SCREENSHOTS_DIR = Path("screenshots")

for folder in [DATA_DIR, PLOTS_DIR, MODELS_DIR, SCREENSHOTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 200

cut_order = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']
color_order = ['J', 'I', 'H', 'G', 'F', 'E', 'D']
clarity_order = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

print("Project folders created and configuration loaded.")

## 2. Download and load the dataset

The project uses the **Diamonds dataset** with **53,940 rows and 10 columns**, as specified in the project document.

The goal here is to:
- download the dataset,
- load it into pandas,
- inspect its basic structure,
- check data quality issues such as nulls and duplicates.

This is the first checkpoint to confirm the dataset matches the project specification.

In [ ]:
data_path = DATA_DIR / "diamonds.csv"

if not data_path.exists():
    gdown.download(GOOGLE_DRIVE_URL, str(data_path), quiet=False)

df = pd.read_csv(data_path)

if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

print("Shape:", df.shape)
display(df.head())
print("info:")
df.info()
print("Describe:")
display(df.describe(include="all"))
print("Null values:")
display(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

## 3. Initial observations

At this stage, I validate the following:
- the dataset has the expected columns,
- categorical columns are correctly identified,
- numerical columns are in usable numeric format,
- there are no major structural issues before preprocessing.

This step is important because reliable modelling starts with reliable data structure.

In [ ]:
print("Columns:", df.columns.tolist())
print("Dtypes:")
print(df.dtypes)

## 4. Data preprocessing

This preprocessing step focuses on three core issues:

### 4.1 Missing values
If any missing values exist:
- numerical values are imputed with median,
- categorical values are imputed with mode.

### 4.2 Invalid zero values in dimensions
The columns `x`, `y`, and `z` represent physical dimensions of a diamond.
A value of zero is not physically meaningful, so these are converted to `NaN` and imputed.

### 4.3 Data type validation
I verify that:
- `cut`, `color`, and `clarity` are categorical/object columns,
- the remaining variables are numeric.

This helps ensure the modelling pipeline remains consistent and avoids silent data issues later.

In [ ]:
df_pre = df.copy()

for col in ['x', 'y', 'z']:
    df_pre[col] = df_pre[col].replace(0, np.nan)

for col in ['x', 'y', 'z']:
    if df_pre[col].isnull().sum() > 0:
        group_median = df_pre.groupby('cut')[col].transform('median')
        df_pre[col] = df_pre[col].fillna(group_median)
        df_pre[col] = df_pre[col].fillna(df_pre[col].median())

numeric_cols = df_pre.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_pre.select_dtypes(include=['object']).columns.tolist()

for col in numeric_cols:
    if df_pre[col].isnull().sum() > 0:
        df_pre[col] = df_pre[col].fillna(df_pre[col].median())

for col in categorical_cols:
    if df_pre[col].isnull().sum() > 0:
        df_pre[col] = df_pre[col].fillna(df_pre[col].mode()[0])

print("Null values after preprocessing:")
display(df_pre.isnull().sum())
print("Dtypes after preprocessing:")
print(df_pre.dtypes)

## 5. Outlier detection and treatment

Luxury pricing datasets often contain true extremes, but very large outliers can destabilize regression performance.

Here, I:
- visualize outliers using boxplots,
- remove extreme values using the **IQR method**,
- and perform a **Z-score check** as a secondary diagnostic.

### Why I used IQR
IQR is robust and interpretable for business datasets because it reduces the influence of extreme values without making too many assumptions about normality.

In [ ]:
outlier_cols_all = ['carat', 'price', 'x', 'y', 'z', 'depth', 'table']

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(outlier_cols_all):
    sns.boxplot(y=df_pre[col], ax=axes[i], color="#d4af37")
    axes[i].set_title(f"Boxplot of {col}")

for j in range(len(outlier_cols_all), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Outlier Boxplots for Core Numerical Features", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "11_outlier_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
def remove_outliers_iqr(dataframe, columns):
    cleaned = dataframe.copy()
    for col in columns:
        q1 = cleaned[col].quantile(0.25)
        q3 = cleaned[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        cleaned = cleaned[(cleaned[col] >= lower) & (cleaned[col] <= upper)]
    return cleaned.reset_index(drop=True)

outlier_cols = ['carat', 'price', 'x', 'y', 'z']
rows_before = len(df_pre)
df_clean = remove_outliers_iqr(df_pre, outlier_cols)
rows_after = len(df_clean)

print(f"Rows before IQR filtering: {rows_before}")
print(f"Rows after IQR filtering : {rows_after}")

z_scores = np.abs(stats.zscore(df_clean[['carat', 'price', 'x', 'y', 'z']]))
print("Rows with |z| > 3 after IQR:", (z_scores > 3).any(axis=1).sum())

## 6. Skewness detection and treatment

Retail price data usually has a long right tail because a small number of premium products are much more expensive than the rest.

To make the regression problem easier to learn, I:
- measure skewness,
- identify highly skewed variables,
- apply `log1p()` transformation where appropriate.

### Why log transformation helps
It compresses extreme ranges, stabilizes variance, and improves model learning for regression tasks involving luxury-price distributions.

In [ ]:
skew_values = df_clean[['carat', 'price', 'x', 'y', 'z', 'depth', 'table']].skew()
print(skew_values)

skewed_cols = ['price', 'carat', 'x', 'y', 'z']
for col in skewed_cols:
    df_clean[f"{col}_log"] = np.log1p(df_clean[col])

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df_clean['price'], kde=True, ax=axes[0, 0], color="#4682b4")
axes[0, 0].set_title("Original Price")

sns.histplot(df_clean['price_log'], kde=True, ax=axes[0, 1], color="#2e8b57")
axes[0, 1].set_title("Log-Transformed Price")

sns.histplot(df_clean['carat'], kde=True, ax=axes[1, 0], color="#b8860b")
axes[1, 0].set_title("Original Carat")

sns.histplot(df_clean['carat_log'], kde=True, ax=axes[1, 1], color="#8a2be2")
axes[1, 1].set_title("Log-Transformed Carat")

plt.suptitle("Skewness Before and After Log Transformation", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "12_skewness_before_after.png", bbox_inches="tight")
plt.show()

## 7. Exploratory Data Analysis (EDA)

EDA helps answer the core business question:

**What actually drives diamond price?**

In this section, I explore:
- price and carat distributions,
- category frequencies,
- price variation by cut, color, and clarity,
- relationships among numerical variables,
- and physical size proxies like volume.

Each chart supports either:
- business interpretation,
- modelling decisions,
- or both.

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_clean['price'], bins=50, kde=True, color="#1f77b4", ax=ax)
ax.set_title("Diamond Price Distribution")
ax.set_xlabel("Price (USD)")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_price_distribution.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
sns.histplot(df_clean['carat'], bins=50, kde=True, color="#c8960c", ax=ax)
ax.set_title("Carat Distribution")
ax.set_xlabel("Carat")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_carat_distribution.png", bbox_inches="tight")
plt.show()

### EDA insight
- Most diamonds lie in lower price bands, with a smaller premium tail.
- Smaller stones dominate the dataset, indicating a highly imbalanced inventory mix.

In [ ]:
for col, fname, title in [
    ('cut', '03_cut_count.png', 'Cut Distribution'),
    ('color', '04_color_count.png', 'Color Distribution'),
    ('clarity', '05_clarity_count.png', 'Clarity Distribution')
]:
    fig, ax = plt.subplots()
    sns.countplot(data=df_clean, x=col, order=df_clean[col].value_counts().index, palette='viridis', ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / fname, bbox_inches="tight")
    plt.show()

### EDA insight
- Category frequencies show how inventory is distributed across quality tiers.
- This is useful for understanding assortment concentration and market positioning.

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(
    data=df_clean.sample(min(10000, len(df_clean)), random_state=RANDOM_STATE),
    x='carat', y='price', alpha=0.35, color="#d4af37", ax=ax
)
ax.set_title("Carat vs Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_price_vs_carat_scatter.png", bbox_inches="tight")
plt.show()

### EDA insight
- Carat is the strongest visible driver of price.
- However, diamonds with similar carat can still have different prices due to quality attributes such as cut, color, and clarity.

In [ ]:
for col, fname, title in [
    ('cut', '07_price_by_cut_boxplot.png', 'Price by Cut'),
    ('color', '08_price_by_color_boxplot.png', 'Price by Color'),
    ('clarity', '09_price_by_clarity_boxplot.png', 'Price by Clarity')
]:
    fig, ax = plt.subplots()
    sns.boxplot(data=df_clean, x=col, y='price', palette='magma', ax=ax)
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / fname, bbox_inches="tight")
    plt.show()

### EDA insight
- Quality categories influence price, but not always in a perfectly monotonic way.
- This indicates feature interaction effects, which supports the use of tree-based and ensemble models.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))
corr = df_clean.select_dtypes(include=[np.number]).corr()
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "10_correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
pairplot_df = df_clean[['carat', 'x', 'y', 'z', 'price']].sample(min(2000, len(df_clean)), random_state=RANDOM_STATE)
sns.pairplot(pairplot_df, diag_kind='kde')
plt.show()

In [ ]:
for feature in ['cut', 'color', 'clarity']:
    df_clean.groupby(feature)['price'].mean().sort_values().plot(kind='bar', color="#d4af37", title=f"Average Price by {feature.capitalize()}")
    plt.ylabel("Average Price (USD)")
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / f"avg_price_by_{feature}.png", bbox_inches="tight")
    plt.show()

In [ ]:
df_clean['volume'] = df_clean['x'] * df_clean['y'] * df_clean['z']

fig, ax = plt.subplots()
sns.scatterplot(
    data=df_clean.sample(min(10000, len(df_clean)), random_state=RANDOM_STATE),
    x='volume', y='price', alpha=0.35, color="#7fffd4", ax=ax
)
ax.set_title("Volume vs Price")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "volume_vs_price.png", bbox_inches="tight")
plt.show()

## 8. Feature engineering

Feature engineering is one of the most important parts of this project because it converts raw data into more business-relevant predictors.

I created the following engineered features:
- `price_inr`
- `volume`
- `price_per_carat`
- `dimension_ratio`
- `carat_category`
- `surface_area`
- `depth_table_ratio`
- `lw_ratio`
- `is_premium`
- `log_price`
- `log_price_inr`

### Why this matters
These features improve the model’s ability to capture:
- size effects,
- shape effects,
- premium quality combinations,
- and business-facing price logic.

In [ ]:
def carat_category(c):
    if c < 0.5:
        return 'Light'
    elif c <= 1.5:
        return 'Medium'
    return 'Heavy'

df_clean['price_inr'] = df_clean['price'] * USD_TO_INR
df_clean['volume'] = df_clean['x'] * df_clean['y'] * df_clean['z']
df_clean['price_per_carat'] = df_clean['price'] / np.clip(df_clean['carat'], 1e-8, None)
df_clean['dimension_ratio'] = (df_clean['x'] + df_clean['y']) / np.clip(2 * df_clean['z'], 1e-8, None)
df_clean['carat_category'] = df_clean['carat'].apply(carat_category)
df_clean['surface_area'] = 2 * (df_clean['x']*df_clean['y'] + df_clean['y']*df_clean['z'] + df_clean['x']*df_clean['z'])
df_clean['depth_table_ratio'] = df_clean['depth'] / np.clip(df_clean['table'], 1e-8, None)
df_clean['lw_ratio'] = df_clean['x'] / np.clip(df_clean['y'], 1e-8, None)

premium_cut = df_clean['cut'] == 'Ideal'
premium_color = df_clean['color'].isin(['D', 'E'])
premium_clarity = df_clean['clarity'].isin(['IF', 'VVS1', 'VVS2'])
df_clean['is_premium'] = (premium_cut & premium_color & premium_clarity).astype(int)

df_clean['log_price'] = np.log1p(df_clean['price'])
df_clean['log_price_inr'] = np.log1p(df_clean['price_inr'])

display(df_clean.head())

## 9. Encoding ordinal categorical features

The columns `cut`, `color`, and `clarity` are ordinal in nature, meaning they have meaningful rank order.

So instead of one-hot encoding, I use **Ordinal Encoding** with domain-specific order:
- `cut`: Fair → Ideal
- `color`: J → D
- `clarity`: I1 → IF

This preserves ranking information, which is especially useful for tree-based models and clustering.

In [ ]:
encoder = OrdinalEncoder(categories=[cut_order, color_order, clarity_order])

df_clean[['cut_enc', 'color_enc', 'clarity_enc']] = encoder.fit_transform(
    df_clean[['cut', 'color', 'clarity']]
)

joblib.dump(encoder, MODELS_DIR / "label_encoders.pkl")
display(df_clean[['cut', 'cut_enc', 'color', 'color_enc', 'clarity', 'clarity_enc']].head())

## 10. Feature selection

To identify the most useful predictors for regression, I use:
- **Random Forest feature importance** as the primary selection method,
- **VIF** as a secondary check for multicollinearity among numeric features.

### Why this is important
Feature selection improves:
- interpretability,
- computational efficiency,
- and sometimes model generalization.

It also makes the final deployed pipeline cleaner and easier to explain.

In [ ]:
feature_cols = [
    'carat', 'cut_enc', 'color_enc', 'clarity_enc',
    'depth', 'table', 'x', 'y', 'z',
    'volume', 'dimension_ratio', 'surface_area',
    'depth_table_ratio', 'lw_ratio', 'is_premium'
]

X_fs = df_clean[feature_cols]
y_fs = df_clean['log_price_inr']

rf_fs = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_fs.fit(X_fs, y_fs)

importances = pd.Series(rf_fs.feature_importances_, index=feature_cols).sort_values(ascending=False)
display(importances)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
importances.sort_values().plot(kind='barh', color="#d4af37", ax=ax)
ax.set_title("Feature Importances")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "13_feature_importances.png", bbox_inches="tight")
plt.show()

selected_features = importances[importances > 0.01].index.tolist()
if len(selected_features) < 5:
    selected_features = importances.head(8).index.tolist()

joblib.dump(selected_features, MODELS_DIR / "selected_features.pkl")
print("Selected Features:", selected_features)

In [ ]:
def compute_vif(dataframe, features):
    vif_df = dataframe[features].dropna().copy()
    vif_values = [variance_inflation_factor(vif_df.values, i) for i in range(vif_df.shape[1])]
    return pd.DataFrame({"feature": features, "VIF": vif_values}).sort_values("VIF", ascending=False)

vif_candidates = [f for f in selected_features if f not in ['cut_enc', 'color_enc', 'clarity_enc', 'is_premium']]
if len(vif_candidates) >= 2:
    vif_report = compute_vif(df_clean, vif_candidates)
    display(vif_report)
else:
    print("Not enough numeric features for VIF calculation.")

## 11. Regression modelling for price prediction

The regression target is:
- **`log_price_inr`**

Why log INR?
- The project requires price prediction in INR.
- Price is highly skewed, so log transformation improves learning stability.

### Models trained
I compare:
- Linear Regression
- Ridge Regression
- Decision Tree Regressor
- Random Forest Regressor
- KNN Regressor
- XGBoost Regressor
- ANN (TensorFlow/Keras)

### Evaluation metrics
I evaluate each model using:
- MAE
- MSE
- RMSE
- R²
- MAPE

All metrics are converted back from log scale to original INR scale for business interpretability.

In [ ]:
X = df_clean[selected_features].copy()
y = df_clean['log_price_inr'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, MODELS_DIR / "scaler_regression.pkl")

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=RANDOM_STATE),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'KNN': KNeighborsRegressor(n_neighbors=5),
    'XGBoost': XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        objective='reg:squarederror'
    )
}

def evaluate_model(y_true_log, y_pred_log, model_name):
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)

    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    mse = mean_squared_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true_orig, y_pred_orig)
    mape = np.mean(np.abs((y_true_orig - y_pred_orig) / np.clip(y_true_orig, 1e-8, None))) * 100

    return {
        'Model': model_name,
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R2': r2,
        'MAPE': mape
    }

In [ ]:
results = []
fitted_models = {}
scaled_models = ['Linear Regression', 'Ridge Regression', 'KNN']

for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name in scaled_models:
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    fitted_models[model_name] = model
    results.append(evaluate_model(y_test, preds, model_name))

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=results_df, x='R2', y='Model', palette='magma', ax=ax)
ax.set_title("Regression Model Comparison by R²")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "14_model_comparison.png", bbox_inches="tight")
plt.show()

## 12. ANN model

To complement the traditional ML models, I also built a feedforward Artificial Neural Network using TensorFlow/Keras.

### ANN architecture
- Dense layers with ReLU activation
- Batch normalization
- Dropout regularization
- Early stopping
- ReduceLROnPlateau

### Why include ANN
Even if a tree-based ensemble becomes the best model, ANN demonstrates:
- deep learning familiarity,
- comparative experimentation,
- and a broader modelling mindset.

In [ ]:
def build_ann(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.30),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.20),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    return model

tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

ann_model = build_ann(X_train_scaled.shape[1])

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=5, verbose=1
)

history = ann_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=256,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

In [ ]:
ann_preds = ann_model.predict(X_test_scaled, verbose=0).flatten()
ann_result = evaluate_model(y_test, ann_preds, "ANN")

results_df = pd.concat([results_df, pd.DataFrame([ann_result])], ignore_index=True)
results_df = results_df.sort_values("R2", ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
history_df = pd.DataFrame(history.history)

fig, ax = plt.subplots(figsize=(12, 7))
ax.plot(history_df['loss'], label='Train Loss', linewidth=2)
ax.plot(history_df['val_loss'], label='Validation Loss', linewidth=2)
ax.set_title("ANN Training History")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / "15_ann_training_history.png", bbox_inches="tight")
plt.show()

## 13. Best model selection and artifact saving

After comparing all models, I save:
- the best regression model,
- the regression scaler,
- the selected features,
- model metrics,
- and metadata for deployment.

This makes the project deployment-ready for the Streamlit application.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
print("Best model:", best_model_name)

if best_model_name == "ANN":
    ann_model.save(MODELS_DIR / "best_ann_model.keras")
    best_model_artifact = "best_ann_model.keras"
    best_model_format = "keras"
else:
    best_model = fitted_models[best_model_name]
    joblib.dump(best_model, MODELS_DIR / "best_regression_model.pkl")
    best_model_artifact = "best_regression_model.pkl"
    best_model_format = "joblib"

results_df.to_csv(MODELS_DIR / "regression_results.csv", index=False)

## 14. Clustering for market segmentation

The second objective of the project is to segment diamonds into market groups.

### Important clustering rule
For clustering, I do **not** use `price` or `price_inr` as input features, because clustering should be based on:
- physical characteristics,
- quality signals,
- and engineered descriptors,

not directly on the target price itself.

### Goal
Find natural market segments that can support:
- merchandising strategy,
- buyer personas,
- inventory tiering,
- and recommendation logic.

In [ ]:
cluster_features = [
    'carat', 'cut_enc', 'color_enc', 'clarity_enc',
    'depth', 'table', 'x', 'y', 'z',
    'volume', 'dimension_ratio', 'surface_area', 'is_premium'
]

X_cluster = df_clean[cluster_features].copy()

cluster_scaler = StandardScaler()
X_cluster_scaled = cluster_scaler.fit_transform(X_cluster)

joblib.dump(cluster_scaler, MODELS_DIR / "scaler_clustering.pkl")

In [ ]:
inertia = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_cluster_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cluster_scaled, labels, sample_size=min(5000, len(df_clean)), random_state=RANDOM_STATE))

print("Inertia:", inertia)
print("Silhouette:", sil_scores)

In [ ]:
fig, ax = plt.subplots()
ax.plot(list(K_range), inertia, marker='o', linewidth=2, color="#d4af37")
ax.set_title("Elbow Method")
ax.set_xlabel("K")
ax.set_ylabel("Inertia")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "16_elbow_method.png", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots()
ax.plot(list(K_range), sil_scores, marker='o', linewidth=2, color="#20b2aa")
ax.set_title("Silhouette Scores")
ax.set_xlabel("K")
ax.set_ylabel("Silhouette Score")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "17_silhouette_scores.png", bbox_inches="tight")
plt.show()

## 15. Fit final KMeans model

The project specification recommends **4 clusters**, so I use `K = 4` for the final segmentation model.

This gives a practical and interpretable grouping suitable for business storytelling.

In [ ]:
optimal_k = 4

kmeans = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
df_clean['cluster'] = kmeans.fit_predict(X_cluster_scaled)

joblib.dump(kmeans, MODELS_DIR / "kmeans_model.pkl")
print(df_clean['cluster'].value_counts().sort_index())

## 16. PCA for cluster visualization

To make high-dimensional clusters interpretable, I reduce the clustering feature space to two principal components.

This does not replace the original clustering model.
It is used only for **visual explanation** and stakeholder communication.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster_scaled)

df_clean['pca1'] = X_pca[:, 0]
df_clean['pca2'] = X_pca[:, 1]

joblib.dump(pca, MODELS_DIR / "pca_model.pkl")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=df_clean.sample(min(15000, len(df_clean)), random_state=RANDOM_STATE),
    x='pca1', y='pca2', hue='cluster', palette='viridis', alpha=0.7, ax=ax
)
ax.set_title("PCA Cluster Projection")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "18_pca_cluster_2d.png", bbox_inches="tight")
plt.show()

## 17. Cluster profiling and naming

Raw cluster IDs like 0, 1, 2, and 3 are not useful for business presentation.

So I profile each cluster using:
- average carat,
- average price,
- average volume,
- encoded cut,
- encoded clarity,
- encoded color,

and then assign human-readable segment names.

In [ ]:
cluster_profile = df_clean.groupby('cluster').agg({
    'carat': 'mean',
    'price': 'mean',
    'volume': 'mean',
    'cut_enc': 'mean',
    'clarity_enc': 'mean',
    'color_enc': 'mean'
}).round(2)

display(cluster_profile)

In [ ]:
cluster_names = {
    0: "Premium Heavy Diamonds",
    1: "Affordable Small Diamonds",
    2: "Mid-Range Balanced Diamonds",
    3: "Quality Light Diamonds"
}

cluster_profile_reset = cluster_profile.reset_index().melt(
    id_vars='cluster', var_name='Feature', value_name='Value'
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.barplot(data=cluster_profile_reset, x='Feature', y='Value', hue='cluster', ax=ax, palette='Set2')
ax.set_title("Cluster Profiles")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "19_cluster_profiles.png", bbox_inches="tight")
plt.show()

## 18. Save processed datasets and metadata

At this point, the modelling work is complete, so I save:
- cleaned and enriched dataset,
- clustered dataset,
- model artifacts,
- metadata JSON.

This is important because the Streamlit app depends on these saved files for prediction and cluster identification.

In [ ]:
df_clean.to_csv(DATA_DIR / "diamond_clean_featured.csv", index=False)

meta = {
    "best_regression_model": best_model_name,
    "best_model_artifact": best_model_artifact,
    "best_model_format": best_model_format,
    "regression_metrics": {
        "R2": round(float(results_df.iloc[0]["R2"]), 4),
        "RMSE_inr": round(float(results_df.iloc[0]["RMSE"]), 2),
        "MAE_inr": round(float(results_df.iloc[0]["MAE"]), 2),
        "MAPE": round(float(results_df.iloc[0]["MAPE"]), 2)
    },
    "ann_metrics": {
        "R2": round(float(results_df[results_df["Model"] == "ANN"].iloc[0]["R2"]), 4) if "ANN" in results_df["Model"].values else None,
        "RMSE_inr": round(float(results_df[results_df["Model"] == "ANN"].iloc[0]["RMSE"]), 2) if "ANN" in results_df["Model"].values else None,
        "MAE_inr": round(float(results_df[results_df["Model"] == "ANN"].iloc[0]["MAE"]), 2) if "ANN" in results_df["Model"].values else None,
        "MAPE": round(float(results_df[results_df["Model"] == "ANN"].iloc[0]["MAPE"]), 2) if "ANN" in results_df["Model"].values else None
    },
    "clustering": {
        "optimal_k": optimal_k,
        "silhouette_score": round(float(silhouette_score(X_cluster_scaled, df_clean['cluster'], sample_size=min(5000, len(df_clean)), random_state=RANDOM_STATE)), 4),
        "cluster_names": {str(k): v for k, v in cluster_names.items()}
    },
    "selected_features": selected_features,
    "cluster_features": cluster_features,
    "usd_to_inr": USD_TO_INR,
    "total_diamonds": int(len(df_clean)),
    "cut_order": cut_order,
    "color_order": color_order,
    "clarity_order": clarity_order
}

with open(MODELS_DIR / "meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

cluster_profile.to_csv(MODELS_DIR / "cluster_profiles.csv")
results_df.to_csv(MODELS_DIR / "regression_results.csv", index=False)

print("Artifacts saved successfully.")

## 19. Final project summary

### What I built
This project delivers:
- a full regression pipeline for diamond price prediction,
- an ANN benchmark model,
- a clustering pipeline for market segmentation,
- PCA-based cluster visualization,
- deployable artifacts for a Streamlit dashboard.

### Key technical highlights
- Invalid physical dimensions handled during preprocessing
- IQR-based outlier filtering
- Log transformation for skewed variables
- Domain-aware feature engineering
- Ordinal encoding for ranked categorical attributes
- Feature importance driven feature selection
- Comparison of classical ML models and ANN
- KMeans clustering with elbow and silhouette analysis
- Cluster naming for business interpretability
- Artifact saving for downstream app deployment

### Business impact
The final system can support:
- retail price estimation,
- product tiering,
- buyer recommendation logic,
- and inventory segmentation.

### Takeaway
This project demonstrates:
- end-to-end machine learning execution,
- strong EDA and feature engineering,
- business storytelling,
- and deployment-oriented thinking.